<div align='center'>
<img src='https://i.imgur.com/9tqpE0m.png' width='80' style='border-radius:50%'/>

# 🎙️ Elbek Studiya — RVC Model Trainer
### Ovoz modeli yaratish vositasi • Gradio + ngrok

---

| Qadam | Nima qiladi |
|:-----:|:------------|
| **1** | Muhitni o'rnatadi (RVC, kutubxonalar) |
| **2** | Gradio + ngrok UI ishga tushiradi |
| **3** | Ovoz yuklang → model nomi kiriting → **Train** |
| **4** | **.pth + .index → ZIP** avtomatik yuklab olinadi |

> ⚡ **GPU tavsiya etiladi:** `Runtime → Change runtime type → T4 GPU`
</div>

In [ ]:
#@title ⚙️ 1-qadam — Muhit o'rnatish { display-mode: "form" }
#@markdown Birinchi marta ishga tushirganda ~5 daqiqa ketadi. Keyingi sessiyalarda tezroq bo'ladi.

import os, sys, subprocess, time

def sh(cmd, check=False):
    print(f'\033[36m$ {cmd}\033[0m')
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout:
        print(r.stdout[-2000:])
    if check and r.returncode != 0:
        raise RuntimeError(f'Xato: {cmd}')
    return r

print('\033[1;33m📦 RVC repozitoriyasi yuklanmoqda...\033[0m')
REPO = '/content/RVC'
if not os.path.exists(REPO):
    sh(f'git clone --depth=1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI {REPO}')
else:
    print('  ⏭️  Repo mavjud, o\'tkazib yuborildi')

os.chdir(REPO)

print('\n\033[1;33m📦 Python kutubxonalari o\'rnatilmoqda...\033[0m')
sh('pip install -q --upgrade pip')
sh('pip install -q -r requirements.txt')
sh('pip install -q pyngrok gradio==3.50.2 praat-parselmouth')
sh('apt-get install -q -y ffmpeg')

print('\n\033[1;33m📥 Pretrained modellar yuklanmoqda...\033[0m')
ASSETS = [
    # (URL, yo'l)
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt',
     'assets/hubert/hubert_base.pt'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt',
     'assets/rmvpe/rmvpe.pt'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/D40k.pth',
     'assets/pretrained_v2/D40k.pth'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/G40k.pth',
     'assets/pretrained_v2/G40k.pth'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth',
     'assets/pretrained_v2/f0D40k.pth'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth',
     'assets/pretrained_v2/f0G40k.pth'),
]
for url, dest in ASSETS:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        sh(f'wget -q --show-progress -O "{dest}" "{url}"')
        print(f'  ✅ {dest}')
    else:
        print(f'  ⏭️  {dest} (mavjud)')

print('\n\033[1;32m✅ Muhit tayyor! 2-qadam ga o\'ting.\033[0m')

In [ ]:
#@title 🚀 2-qadam — Gradio UI + ngrok ishga tushirish { display-mode: "form" }
#@markdown ### ngrok tokeni (ixtiyoriy — bepul havolani olish uchun https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = '' #@param {type:"string"}
#@markdown ---
#@markdown > **ngrok tokeni bo'lmasa ham ishlaydi** — Gradio `.gradio.live` havolasini avtomatik beradi.

import gradio as gr
import os, shutil, zipfile, glob, subprocess, threading, re, time
from pathlib import Path

BASE = '/content/RVC'
os.chdir(BASE)

# ─── ngrok sozlash ──────────────────────────────────────────────────────────
PUBLIC_URL = None
if NGROK_TOKEN and NGROK_TOKEN.strip():
    try:
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = NGROK_TOKEN.strip()
        ngrok.kill()  # avvalgi tunnelni to'xtatish
        tunnel = ngrok.connect(7860, proto='http')
        PUBLIC_URL = tunnel.public_url
        print(f'\033[1;32m🔗 ngrok URL: {PUBLIC_URL}\033[0m')
    except Exception as e:
        print(f'⚠️  ngrok xato: {e} — Gradio share ishlatiladi')

# ─── yordamchi funksiyalar ───────────────────────────────────────────────────

def clean_name(s):
    return re.sub(r'[^\w\-]', '_', s.strip()) if s else ''

def find_pth_index(model_name):
    pth   = glob.glob(f'{BASE}/logs/{model_name}/*.pth')
    pth  += glob.glob(f'{BASE}/weights/{model_name}*.pth')
    idx   = glob.glob(f'{BASE}/logs/{model_name}/*.index')
    return list(set(pth)), list(set(idx))

def create_zip(model_name, pth_files, idx_files):
    zip_path = f'/content/{model_name}_ElbekStudiya.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in pth_files:
            zf.write(f, os.path.basename(f))
        for f in idx_files:
            zf.write(f, os.path.basename(f))
    return zip_path

def run_cmd(cmd, cwd=BASE):
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=cwd)
    return (r.stdout or '')[:4000], r.returncode

# ─── asosiy o'qitish funksiyasi ──────────────────────────────────────────────

def train_model(
    model_name_raw,
    audio_files,
    total_epoch,
    sample_rate,
    pitch_algo,
    batch_size,
    save_every,
    progress=gr.Progress(track_tqdm=True)
):
    logs = []
    def L(msg, color=''):
        logs.append(msg)
        return '\n'.join(logs)

    if not model_name_raw or not model_name_raw.strip():
        return '❌ Model nomini kiriting!', None
    if not audio_files:
        return '❌ Audio fayl yuklang!', None

    name = clean_name(model_name_raw)
    L(f'🎯 Model nomi : {name}')
    L(f'📁 Fayl soni  : {len(audio_files)}')
    L(f'🔢 Epoch      : {total_epoch}')
    L(f'🎵 Pitch algo : {pitch_algo}')
    L('─'*50)

    try:
        # 1) Dataset tayyorlash
        progress(0.05, desc='📁 Dataset tayyorlanmoqda...')
        ds_dir = f'{BASE}/dataset/{name}'
        if os.path.exists(ds_dir):
            shutil.rmtree(ds_dir)
        os.makedirs(ds_dir, exist_ok=True)
        for f in audio_files:
            shutil.copy(f.name, os.path.join(ds_dir, os.path.basename(f.name)))
        L(f'✅ Dataset: {ds_dir}')

        log_dir = f'{BASE}/logs/{name}'
        os.makedirs(log_dir, exist_ok=True)

        # 2) Preprocessing
        progress(0.12, desc='🔊 Ovoz qayta ishlanmoqda...')
        L('\n⏳ [1/5] Ovoz qayta ishlanmoqda...')
        out, rc = run_cmd(
            f'python trainset_preprocess_pipeline_print.py "{ds_dir}" {sample_rate} 2 "{log_dir}" False'
        )
        L(out[-1500:] if out else '(natija yo\'q)')
        if rc != 0:
            L('⚠️  Preprocessing xato (davom etiladi)')

        # 3) Pitch extraction
        progress(0.28, desc=f'🎵 Pitch chiqarilmoqda ({pitch_algo})...')
        L(f'\n⏳ [2/5] Pitch chiqarilmoqda ({pitch_algo})...')
        out, rc = run_cmd(
            f'python extract_f0_print.py "{log_dir}" 2 "{pitch_algo}"'
        )
        L(out[-1500:] if out else '(natija yo\'q)')

        # 4) Feature extraction
        progress(0.44, desc='🧠 Xususiyatlar chiqarilmoqda...')
        L('\n⏳ [3/5] Xususiyatlar chiqarilmoqda...')
        out, rc = run_cmd(
            f'python extract_feature_print.py cuda:0 1 0 0 "{log_dir}" v2'
        )
        L(out[-1500:] if out else '(natija yo\'q)')

        # 5) Training
        progress(0.55, desc=f'🏋️ O\'qitish {total_epoch} epoch...')
        L(f'\n⏳ [4/5] O\'qitish ({total_epoch} epoch, batch={batch_size})...')
        L('⏳ Bu biroz vaqt olishi mumkin, kuting...')
        out, rc = run_cmd(
            f'python train_nsf_sim_cache_sid_load_pretrain.py '
            f'-e "{name}" -sr {sample_rate} -f0 1 -bs {batch_size} '
            f'-g 0 -te {total_epoch} -se {save_every} '
            f'-pg assets/pretrained_v2/f0G40k.pth '
            f'-pd assets/pretrained_v2/f0D40k.pth '
            f'-l 1 -c 0 -sw 0 -v v2'
        )
        L(out[-3000:] if out else '(natija yo\'q)')
        if rc != 0:
            L(f'⚠️  O\'qitish xatosi (rc={rc})')

        # 6) Index building
        progress(0.88, desc='📑 Index yaratilmoqda...')
        L('\n⏳ [5/5] Index yaratilmoqda...')
        out, rc = run_cmd(f'python train_index.py "{log_dir}" v2')
        L(out[-1500:] if out else '(natija yo\'q)')

        # 7) ZIP
        progress(0.95, desc='📦 ZIP yaratilmoqda...')
        pth_files, idx_files = find_pth_index(name)
        if not pth_files and not idx_files:
            L('\n❌ Chiqish fayllari topilmadi!')
            L(f'   Qidirilgan joylari: {BASE}/logs/{name}/ va {BASE}/weights/')
            return '\n'.join(logs), None

        zip_path = create_zip(name, pth_files, idx_files)
        L('\n' + '='*50)
        L('🎉 O\'QITISH MUVAFFAQIYATLI YAKUNLANDI!')
        L('='*50)
        L(f'📦 ZIP: {zip_path}')
        L(f'   PTH   : {[os.path.basename(f) for f in pth_files]}')
        L(f'   INDEX : {[os.path.basename(f) for f in idx_files]}')
        L('⬇️  Quyidagi "Yuklab olish" tugmasini bosing!')

        progress(1.0, desc='🎉 Tayyor!')
        return '\n'.join(logs), zip_path

    except Exception:
        import traceback
        L(f'\n💥 Xato:\n{traceback.format_exc()}')
        return '\n'.join(logs), None

# ─── Gradio UI ───────────────────────────────────────────────────────────────

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');

* { font-family: 'Inter', sans-serif !important; }

.es-header {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 40%, #0f3460 100%);
    border-radius: 16px;
    padding: 28px 20px;
    text-align: center;
    margin-bottom: 20px;
    border: 1px solid #e94560;
    box-shadow: 0 8px 32px rgba(233,69,96,0.3);
}
.es-header h1 {
    color: #ffffff;
    font-size: 2.2em;
    font-weight: 700;
    margin: 0;
    letter-spacing: 1px;
}
.es-header .accent { color: #e94560; }
.es-header p {
    color: #a8b2d8;
    margin: 8px 0 0 0;
    font-size: 0.95em;
}
.es-badge {
    display:inline-block;
    background:#e94560;
    color:#fff;
    border-radius:20px;
    padding:3px 12px;
    font-size:0.75em;
    font-weight:600;
    margin-top:8px;
}
.panel-left  { background:#0d1117; border-radius:12px; padding:16px; border:1px solid #21262d; }
.panel-right { background:#0d1117; border-radius:12px; padding:16px; border:1px solid #21262d; }
label { color: #cdd9e5 !important; font-weight: 600 !important; }
.gr-button-primary {
    background: linear-gradient(135deg, #e94560, #c62a47) !important;
    border: none !important;
    border-radius: 10px !important;
    font-size: 1.1em !important;
    font-weight: 700 !important;
    padding: 14px !important;
    box-shadow: 0 4px 15px rgba(233,69,96,0.4) !important;
    transition: all 0.3s !important;
}
.gr-button-primary:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 25px rgba(233,69,96,0.5) !important;
}
.footer-note {
    text-align:center;
    color:#8b949e;
    font-size:0.82em;
    margin-top:16px;
    padding:12px;
    border-top:1px solid #21262d;
}
"""

HEADER_HTML = """
<div class='es-header'>
  <h1>🎙️ <span class='accent'>Elbek Studiya</span> RVC Model Trainer</h1>
  <p>Ovozingizni yuklang · Model nomini kiriting · Train tugmasini bosing</p>
  <span class='es-badge'>⚡ Retrieval-based Voice Conversion v2</span>
</div>
"""

with gr.Blocks(css=CSS, title='Elbek Studiya — RVC Trainer', theme=gr.themes.Base(
    primary_hue=gr.themes.colors.red,
    secondary_hue=gr.themes.colors.blue,
    neutral_hue=gr.themes.colors.slate,
    font=gr.themes.GoogleFont('Inter')
).set(
    body_background_fill='#010409',
    block_background_fill='#0d1117',
    block_border_color='#21262d',
    input_background_fill='#161b22',
    button_primary_background_fill='#e94560',
    button_primary_background_fill_hover='#c62a47',
)) as demo:

    gr.HTML(HEADER_HTML)

    with gr.Row(equal_height=True):

        # ── Chap panel: Kirish ─────────────────────────────────────────────
        with gr.Column(scale=5):
            gr.Markdown('### 📋 Model sozlamalari')

            with gr.Group():
                model_name_inp = gr.Textbox(
                    label='🏷️ Model nomi',
                    placeholder='masalan: elbek_ovozi  yoki  sarvar_2024',
                    max_lines=1,
                    info='Faqat lotin harflari, raqamlar va _ belgisi'
                )

                audio_inp = gr.File(
                    label='🎤 Ovoz fayllari  (.wav / .mp3 / .flac)',
                    file_count='multiple',
                    file_types=['.wav','.mp3','.flac','.ogg','.m4a'],
                    interactive=True
                )

            with gr.Accordion('⚙️ Kengaytirilgan sozlamalar', open=False):
                with gr.Row():
                    epoch_inp = gr.Slider(
                        label='🔢 Epoch soni',
                        minimum=10, maximum=1000, value=100, step=10,
                        info='Ko\'proq epoch = yaxshiroq sifat (lekin ko\'proq vaqt)'
                    )
                    save_every_inp = gr.Slider(
                        label='💾 Har necha epochda saqlash',
                        minimum=5, maximum=100, value=10, step=5
                    )
                with gr.Row():
                    sr_inp = gr.Radio(
                        label='📻 Sample Rate',
                        choices=['40000','48000'],
                        value='40000',
                        info='40k — standart, 48k — yuqori sifat'
                    )
                    pitch_inp = gr.Dropdown(
                        label='🎵 Pitch algoritmi',
                        choices=['rmvpe','harvest','crepe','pm'],
                        value='rmvpe',
                        info='rmvpe — eng aniq (tavsiya etiladi)'
                    )
                batch_inp = gr.Slider(
                    label='📦 Batch size',
                    minimum=1, maximum=24, value=4, step=1,
                    info='GPU xotirasi kam bo\'lsa 2-4, ko\'p bo\'lsa 8-16'
                )

            train_btn = gr.Button(
                '🚀  O\'QITISHNI BOSHLASH',
                variant='primary',
                size='lg'
            )

            # Tezkor yo'riqnoma
            gr.Markdown("""
---
**📌 Qo'llanma:**
1. Model nomini kiriting (masalan: `sarvar_ovozi`)
2. `.wav`/`.mp3` ovoz fayllarini yuklang (10+ daqiqa tavsiya)
3. Epoch sonini tanlang (100–300 tavsiya)
4. **O'qitishni boshlash** tugmasini bosing
5. Tayyor bo'lgach, **ZIP fayl** yuklab oling 🎉
""")

        # ── O'ng panel: Natija ─────────────────────────────────────────────
        with gr.Column(scale=5):
            gr.Markdown('### 📊 Natija va jarayon')

            log_out = gr.Textbox(
                label='📜 Jarayon jurnali',
                lines=24,
                interactive=False,
                show_copy_button=True,
                placeholder=(
                    'O\'qitish boshlanganidan keyin bu yerda jarayon ko\'rinadi...\n\n'
                    '  ✅ Dataset tayyor\n'
                    '  ⏳ Ovoz qayta ishlanmoqda...\n'
                    '  🎵 Pitch chiqarilmoqda...\n'
                    '  🧠 Xususiyatlar chiqarilmoqda...\n'
                    '  🏋️ Model o\'qitilmoqda...\n'
                    '  📦 ZIP arxivlanmoqda...\n'
                    '  🎉 Tayyor!'
                )
            )

            zip_out = gr.File(
                label='📦 Tayyor model ZIP (yuklab olish)',
                interactive=False
            )

            gr.Markdown("""
---
**📦 ZIP tarkibi:**
- `model_nomi_eXXX.pth` — Asosiy model fayli
- `added_IVF_model.index` — Qidiruv indeksi

**🔗 Foydalanish mumkin bo'lgan platformalar:**
[RVC Simple](https://huggingface.co/spaces/r3gm/RVC_HF) • [AICoverGen](https://github.com/SociallyIneptWeeb/AICoverGen) • [w-okam](https://w-okam.com)
""")

    gr.HTML("""
<div class='footer-note'>
  🎙️ <strong>Elbek Studiya RVC Trainer</strong> ·
  Powered by <a href='https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI'
  style='color:#e94560'>RVC-Project</a> ·
  UI: <a href='https://gradio.app' style='color:#e94560'>Gradio</a>
</div>
""")

    train_btn.click(
        fn=train_model,
        inputs=[model_name_inp, audio_inp, epoch_inp, sr_inp,
                pitch_inp, batch_inp, save_every_inp],
        outputs=[log_out, zip_out],
        show_progress=True
    )

# ─── ishga tushirish ─────────────────────────────────────────────────────────
print('\n\033[1;32m🚀 Gradio UI ishga tushmoqda...\033[0m')
demo.queue(max_size=5).launch(
    server_port=7860,
    share=True,       # gradio.live havolasini yaratadi
    debug=False,
    show_error=True,
    quiet=False,
    inbrowser=False
)